# ISPRA RU costi e kg per abitante - v0

## Domanda

I territori che producono meno rifiuti urbani o raccolgono meglio spendono anche meno per abitante?

Questo `v0` non prova ancora a rispondere sull'intero universo dei comuni italiani. Lavora sul perimetro più stretto in cui il join tra:

- RU base
- costi per chilogrammo
- costi per abitante

regge davvero sullo stesso `codice_comune_istat x anno`.

L'obiettivo di questa prima lettura è triplice:

1. misurare la copertura del join `A + B + C` tra `2020` e `2024`
2. osservare il perimetro joinato `2024` come primo anno di lettura pubblica
3. capire se esiste già un segnale leggibile tra `kg RU per abitante`, `costo per abitante` e `% raccolta differenziata`


In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


def _find_repo_root() -> Path:
    """Cerca il root di dataset-incubator risalendo dai parents del CWD."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'candidates').exists() and (p / 'out').exists():
            return p
    # fallback: notebook eseguito dalla sua directory
    return Path.cwd().parents[2]


ROOT = _find_repo_root()
BASE = ROOT / 'out' / 'data' / 'mart' / 'ispra_ru_base'
YEARS = [2020, 2021, 2022, 2023, 2024]

con = duckdb.connect()

# Usa solo gli anni con mart_cross_comuni.parquet effettivamente disponibili
available_years = sorted([
    int(p.parent.name)
    for p in BASE.glob('*/mart_cross_comuni.parquet')
])
print(f'Anni disponibili con cross mart: {available_years}')

paths = [(BASE / str(year) / 'mart_cross_comuni.parquet').as_posix() for year in available_years]
union_sql = " UNION ALL ".join([f"SELECT * FROM read_parquet('" + path + "')" for path in paths])
all_years_sql = f"with all_years as ({union_sql}) select * from all_years"


## Copertura del join `A + B + C`

Prima di leggere costi e performance, conviene capire su quanti comuni il cross regge davvero. Questa è la soglia metodologica più importante del filone in questa fase.


In [ ]:
coverage = con.execute(f"""
with all_years as ({union_sql})
select
    anno,
    count(*) as comuni_ru,
    sum(case when join_b_ok and join_c_ok then 1 else 0 end) as comuni_join,
    round(100.0 * sum(case when join_b_ok and join_c_ok then 1 else 0 end) / count(*), 2) as copertura_join_pct
from all_years
group by 1
order by 1
""").fetchdf()
coverage

La copertura migliora molto tra `2020` e `2024`: il perimetro joinato passa da poco meno del `58%` a oltre l'`84%` dei comuni presenti nel dataset RU base. Questo basta per una prima lettura, ma non ancora per trattare il cross come universo completo dei comuni italiani.


## Focus 2024 sul perimetro joinato

Per il primo output conviene fermarsi all'ultimo anno disponibile e lavorare solo sui comuni che hanno insieme RU base, costi per kg e costi per abitante. È una scelta voluta: prima testiamo se il cross regge bene sul presente, poi valutiamo se estendere il racconto lungo tutta la serie `2020-2024`.


In [ ]:
joined_2024 = con.execute(f"""
with all_years as ({union_sql})
select *
from all_years
where anno = 2024
  and join_b_ok
  and join_c_ok
""").fetchdf()

summary_2024 = pd.DataFrame([
    {'metrica': 'Comuni nel perimetro joinato 2024', 'valore': int(len(joined_2024))},
    {'metrica': 'Kg RU per abitante medi', 'valore': round(joined_2024['kg_ru_per_abitante_calc'].mean(), 2)},
    {'metrica': 'Costo totale medio per abitante (euro)', 'valore': round(joined_2024['costo_totale_euro_ab'].mean(), 2)},
    {'metrica': 'Percentuale RD media', 'valore': round(joined_2024['percentuale_rd'].mean(), 2)},
])
summary_2024

In [ ]:
corr = joined_2024[['kg_ru_per_abitante_calc', 'costo_totale_euro_ab', 'percentuale_rd']].corr().round(3)
corr


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sample = joined_2024.sample(min(len(joined_2024), 2500), random_state=42)

axes[0].scatter(sample['kg_ru_per_abitante_calc'], sample['costo_totale_euro_ab'], alpha=0.25, s=12)
axes[0].set_title('2024: kg RU per abitante vs costo per abitante')
axes[0].set_xlabel('kg RU per abitante')
axes[0].set_ylabel('costo totale euro per abitante')

axes[1].scatter(sample['percentuale_rd'], sample['costo_totale_euro_ab'], alpha=0.25, s=12, color='darkgreen')
axes[1].set_title('2024: % raccolta differenziata vs costo per abitante')
axes[1].set_xlabel('% raccolta differenziata')
axes[1].set_ylabel('costo totale euro per abitante')

plt.tight_layout()
plt.show()


La prima immagine mostra un segnale abbastanza leggibile: all'aumentare dei `kg RU per abitante`, tende ad aumentare anche il `costo per abitante`. La seconda relazione è meno netta: la `% RD` non sembra spiegare da sola il livello di costo, almeno non in questo perimetro e senza altri controlli.


## Alcuni outlier leggibili

In questa fase conviene guardare pochi casi, senza trattarli ancora come ranking definitivo. I tre tagli qui sotto servono a capire se il cross produce profili territoriali plausibili:

- i comuni con costo per abitante molto alto
- i comuni con costi alti anche a fronte di `kg` relativamente bassi
- i comuni con `% RD` alta e costo relativamente contenuto


In [ ]:
top_costi = joined_2024[
    ['regione', 'provincia', 'comune', 'kg_ru_per_abitante_calc', 'percentuale_rd', 'costo_totale_euro_ab']
].sort_values('costo_totale_euro_ab', ascending=False).head(10)

top_costi


In [ ]:
alti_costi_bassi_kg = joined_2024[
    (joined_2024['kg_ru_per_abitante_calc'] < 400) &
    (joined_2024['costo_totale_euro_ab'] > joined_2024['costo_totale_euro_ab'].quantile(0.95))
][['regione', 'provincia', 'comune', 'kg_ru_per_abitante_calc', 'percentuale_rd', 'costo_totale_euro_ab']].sort_values('costo_totale_euro_ab', ascending=False).head(10)

alti_costi_bassi_kg


In [ ]:
rd_alta_costo_basso = joined_2024[
    (joined_2024['percentuale_rd'] >= 80) &
    (joined_2024['costo_totale_euro_ab'] <= joined_2024['costo_totale_euro_ab'].quantile(0.35))
][['regione', 'provincia', 'comune', 'kg_ru_per_abitante_calc', 'percentuale_rd', 'costo_totale_euro_ab']].sort_values(['percentuale_rd', 'costo_totale_euro_ab'], ascending=[False, True]).head(10)

rd_alta_costo_basso


## Prime evidenze

- il cross `A + B + C` è già usabile, ma solo su un perimetro joinato esplicito
- tra `2020` e `2024` la copertura del join migliora molto
- nel perimetro joinato `2024`, `kg RU per abitante` e `costo per abitante` si muovono nella stessa direzione con una relazione visibile anche prima di modelli più forti
- la `% RD` aggiunge informazione, ma qui non sembra bastare da sola per spiegare i costi
- i casi estremi suggeriscono che una futura `v1` dovrebbe distinguere meglio profili territoriali diversi, non solo fare ranking

## Nota metodologica minima

Questa lettura non copre ancora tutto il perimetro RU comunale. Lavora solo sui comuni per cui il join tra dataset base, costi per kg e costi per abitante riesce davvero. I risultati vanno quindi letti come primo test del cross, non come fotografia completa di tutti i comuni italiani.
